In [8]:
from datasets import load_dataset
dataset = load_dataset("UBC-SLIME/colx_531_group_project")

In [9]:
from transformers import AutoModelForCausalLM

lm1 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-135m")

lm2 = AutoModelForCausalLM.from_pretrained("UBC-SLIME/colx_531_smollm2-360m")

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 18714.49it/s]


In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 50000
    })
    validation: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['id', 'text', 'is_member'],
        num_rows: 15000
    })
})

In [11]:
import pandas as pd
import altair as alt
import numpy as np
from collections import Counter
import re
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import scipy.sparse as sp

# Configure Altair
alt.data_transformers.enable('json')

DataTransformerRegistry.enable('json')

# Membership Inference Attack - EDA

**Goal:** Identify "memorization signatures" that distinguish texts used to fine-tune the models (members) from those that weren't (non-members).

We'll focus on patterns that could indicate memorization rather than generic text statistics.

In [12]:
# Quick Dataset Overview
print(f"Training set: {len(dataset['train'])} samples")
print(f"Class balance: {dataset['train'].to_pandas()['is_member'].value_counts().to_dict()}")

# Convert to pandas for analysis
train_df = dataset['train'].to_pandas()
print(f"\\nSample record:")
print(f"Text preview: {train_df.iloc[0]['text'][:200]}...")
print(f"Is member: {train_df.iloc[0]['is_member']}")

Training set: 50000 samples
Class balance: {0: 25000, 1: 25000}
\nSample record:
Text preview: Discharge Summary:
Patient Name: N/A
Patient ID: N/A
Gender: Female
Age: 43
Admit Date: 29th January 2018
Discharge Date: N/A

Chief Complaint:
Infertility evaluation and management.

Hospital Course ...
Is member: 0


In [13]:
# Quick Class Distribution Check
class_counts = train_df['is_member'].value_counts()
print(f"Class distribution: {class_counts.to_dict()}")
print(f"Dataset is {'balanced' if abs(class_counts[0] - class_counts[1]) < 1000 else 'imbalanced'}")

# Split data for subsequent analysis
member_texts = train_df[train_df['is_member'] == 1]
non_member_texts = train_df[train_df['is_member'] == 0]
print(f"\\nMember texts: {len(member_texts)}, Non-member texts: {len(non_member_texts)}")

Class distribution: {0: 25000, 1: 25000}
Dataset is balanced
\nMember texts: 25000, Non-member texts: 25000


# EDA for Membership Inference

The goal is to find "memorization signatures" that distinguish training data from non-training data.

In [14]:
# 7. Text Uniqueness and Rarity Analysis

def analyze_text_uniqueness(df):
    """Analyze text uniqueness and rarity patterns"""
    
    # Calculate text-level metrics
    df = df.copy()
    df['text_lower'] = df['text'].str.lower()
    
    # Exact duplicates
    df['is_duplicate'] = df.duplicated(subset=['text_lower'], keep=False)
    
    # Near duplicates (simple Jaccard similarity for substrings)
    def get_char_ngrams(text, n=4):
        return set(text[i:i+n] for i in range(len(text)-n+1))
    
    def jaccard_similarity(text1, text2, n=4):
        ngrams1 = get_char_ngrams(text1, n)
        ngrams2 = get_char_ngrams(text2, n)
        if len(ngrams1) == 0 and len(ngrams2) == 0:
            return 1.0
        return len(ngrams1.intersection(ngrams2)) / len(ngrams1.union(ngrams2))
    
    # Calculate vocabulary richness
    df['unique_words'] = df['text_lower'].apply(lambda x: len(set(x.split())))
    df['total_words'] = df['text_lower'].apply(lambda x: len(x.split()))
    df['type_token_ratio'] = df['unique_words'] / df['total_words']
    
    # Calculate rare word frequency using TF-IDF
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(df['text_lower'])
    
    # Get high TF-IDF scores (potentially rare/unique phrases)
    max_tfidf_scores = np.array(tfidf_matrix.max(axis=1).todense()).flatten()
    df['max_tfidf'] = max_tfidf_scores
    
    # Mean TF-IDF (overall rarity)
    mean_tfidf_scores = np.asarray(tfidf_matrix.mean(axis=1)).flatten()
    df['mean_tfidf'] = mean_tfidf_scores
    
    return df

# Analyze uniqueness patterns
train_df_enhanced = analyze_text_uniqueness(train_df)

# Compare member vs non-member uniqueness
member_data = train_df_enhanced[train_df_enhanced['is_member'] == 1]
non_member_data = train_df_enhanced[train_df_enhanced['is_member'] == 0]

print("Text Uniqueness Analysis:")
print("=" * 50)
print(f"\\nDuplicates:")
print(f"  Member texts with duplicates: {member_data['is_duplicate'].sum()}/{len(member_data)} ({member_data['is_duplicate'].mean()*100:.1f}%)")
print(f"  Non-member texts with duplicates: {non_member_data['is_duplicate'].sum()}/{len(non_member_data)} ({non_member_data['is_duplicate'].mean()*100:.1f}%)")

print(f"\\nVocabulary Richness (Type-Token Ratio):")
print(f"  Member texts: {member_data['type_token_ratio'].mean():.3f} ± {member_data['type_token_ratio'].std():.3f}")
print(f"  Non-member texts: {non_member_data['type_token_ratio'].mean():.3f} ± {non_member_data['type_token_ratio'].std():.3f}")

print(f"\\nRare Content (Max TF-IDF):")
print(f"  Member texts: {member_data['max_tfidf'].mean():.3f} ± {member_data['max_tfidf'].std():.3f}")
print(f"  Non-member texts: {non_member_data['max_tfidf'].mean():.3f} ± {non_member_data['max_tfidf'].std():.3f}")

# Visualize uniqueness metrics
uniqueness_data = pd.concat([
    pd.DataFrame({
        'type_token_ratio': member_data['type_token_ratio'],
        'max_tfidf': member_data['max_tfidf'],
        'mean_tfidf': member_data['mean_tfidf'],
        'class': 'Member'
    }),
    pd.DataFrame({
        'type_token_ratio': non_member_data['type_token_ratio'],
        'max_tfidf': non_member_data['max_tfidf'],
        'mean_tfidf': non_member_data['mean_tfidf'],
        'class': 'Non-Member'
    })
]).reset_index(drop=True)

# Create comparison charts
ttr_chart = alt.Chart(uniqueness_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('type_token_ratio:Q', title='Type-Token Ratio'),
    color=alt.Color('class:N')
).properties(title='Vocabulary Richness by Class', width=250, height=200)

tfidf_chart = alt.Chart(uniqueness_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('max_tfidf:Q', title='Max TF-IDF Score'),
    color=alt.Color('class:N')
).properties(title='Content Rarity by Class', width=250, height=200)

(ttr_chart | tfidf_chart).show()

Text Uniqueness Analysis:
\nDuplicates:
  Member texts with duplicates: 594/25000 (2.4%)
  Non-member texts with duplicates: 667/25000 (2.7%)
\nVocabulary Richness (Type-Token Ratio):
  Member texts: 0.576 ± 0.071
  Non-member texts: 0.577 ± 0.071
\nRare Content (Max TF-IDF):
  Member texts: 0.278 ± 0.073
  Non-member texts: 0.279 ± 0.074


alt.HConcatChart(...)

In [15]:
# 8. Medical Domain-Specific Analysis
def analyze_medical_patterns(df):
    """Analyze medical terminology and discharge note structure"""
    
    df = df.copy()
    
    # Common medical terms/patterns (basic medical vocabulary)
    medical_terms = [
        'patient', 'diagnosis', 'treatment', 'medication', 'discharge', 'admission',
        'symptoms', 'condition', 'procedure', 'hospital', 'doctor', 'nurse',
        'blood', 'pressure', 'heart', 'lung', 'kidney', 'liver', 'brain',
        'surgery', 'therapy', 'pain', 'fever', 'infection', 'chronic', 'acute',
        'mg', 'ml', 'dose', 'daily', 'twice', 'prescribed', 'follow-up',
    ]
    
    # Structured sections common in discharge notes
    structured_patterns = [
        'discharge summary', 'chief complaint', 'history of present illness',
        'past medical history', 'medications', 'allergies', 'physical exam',
        'assessment', 'plan', 'disposition', 'follow up', 'instructions'
    ]
    
    # Calculate medical term density
    text_lower = df['text'].str.lower()
    df['medical_term_count'] = 0
    for term in medical_terms:
        df['medical_term_count'] += text_lower.str.count(term)
    
    df['medical_term_density'] = df['medical_term_count'] / df['text'].str.split().str.len()
    
    # Calculate structured section count
    df['structured_section_count'] = 0
    for pattern in structured_patterns:
        df['structured_section_count'] += text_lower.str.count(pattern)
    
    # Numeric pattern analysis (medical values, dates, etc.)
    df['numeric_count'] = text_lower.str.count(r'\\d+')
    df['date_pattern_count'] = text_lower.str.count(r'\\d{1,2}/\\d{1,2}/\\d{2,4}')
    df['time_pattern_count'] = text_lower.str.count(r'\\d{1,2}:\\d{2}')
    
    # Medication pattern count (basic patterns)
    df['medication_pattern_count'] = (
        text_lower.str.count(' mg') + 
        text_lower.str.count(' ml') + 
        text_lower.str.count(' mcg') +
        text_lower.str.count('daily') +
        text_lower.str.count('twice') +
        text_lower.str.count('q\\d+h')  # q8h, q12h patterns
    )
    
    # Capitalized words (could indicate medical acronyms/abbreviations)
    df['caps_word_count'] = df['text'].str.count(r'\\b[A-Z]{2,}\\b')
    
    return df

# Analyze medical patterns
train_df_medical = analyze_medical_patterns(train_df)

# Compare medical characteristics by class
member_medical = train_df_medical[train_df_medical['is_member'] == 1]
non_member_medical = train_df_medical[train_df_medical['is_member'] == 0]

print("Medical Domain Analysis:")
print("=" * 50)
print(f"\\nMedical Term Density:")
print(f"  Member texts: {member_medical['medical_term_density'].mean():.3f} ± {member_medical['medical_term_density'].std():.3f}")
print(f"  Non-member texts: {non_member_medical['medical_term_density'].mean():.3f} ± {non_member_medical['medical_term_density'].std():.3f}")

print(f"\\nStructured Sections:")
print(f"  Member texts: {member_medical['structured_section_count'].mean():.2f} ± {member_medical['structured_section_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['structured_section_count'].mean():.2f} ± {non_member_medical['structured_section_count'].std():.2f}")

print(f"\\nMedical Abbreviations (CAPS):")
print(f"  Member texts: {member_medical['caps_word_count'].mean():.2f} ± {member_medical['caps_word_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['caps_word_count'].mean():.2f} ± {non_member_medical['caps_word_count'].std():.2f}")

print(f"\\nMedication Patterns:")
print(f"  Member texts: {member_medical['medication_pattern_count'].mean():.2f} ± {member_medical['medication_pattern_count'].std():.2f}")
print(f"  Non-member texts: {non_member_medical['medication_pattern_count'].mean():.2f} ± {non_member_medical['medication_pattern_count'].std():.2f}")

# Visualize medical characteristics
medical_data = pd.concat([
    pd.DataFrame({
        'medical_term_density': member_medical['medical_term_density'],
        'structured_section_count': member_medical['structured_section_count'],
        'caps_word_count': member_medical['caps_word_count'],
        'medication_pattern_count': member_medical['medication_pattern_count'],
        'class': 'Member'
    }),
    pd.DataFrame({
        'medical_term_density': non_member_medical['medical_term_density'],
        'structured_section_count': non_member_medical['structured_section_count'],
        'caps_word_count': non_member_medical['caps_word_count'],
        'medication_pattern_count': non_member_medical['medication_pattern_count'],
        'class': 'Non-Member'
    })
]).reset_index(drop=True)

# Create medical pattern comparison charts
med_density_chart = alt.Chart(medical_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('medical_term_density:Q', title='Medical Term Density'),
    color=alt.Color('class:N')
).properties(title='Medical Term Usage', width=200, height=180)

structure_chart = alt.Chart(medical_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('structured_section_count:Q', title='Structured Sections'),
    color=alt.Color('class:N')
).properties(title='Document Structure', width=200, height=180)

caps_chart = alt.Chart(medical_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('caps_word_count:Q', title='Abbreviation Count'),
    color=alt.Color('class:N')
).properties(title='Medical Abbreviations', width=200, height=180)

medication_chart = alt.Chart(medical_data).mark_boxplot().encode(
    x=alt.X('class:N', title='Class'),
    y=alt.Y('medication_pattern_count:Q', title='Medication Patterns'),
    color=alt.Color('class:N')
).properties(title='Medication References', width=200, height=180)

((med_density_chart | structure_chart) & (caps_chart | medication_chart)).show()

Medical Domain Analysis:
\nMedical Term Density:
  Member texts: 0.087 ± 0.035
  Non-member texts: 0.087 ± 0.035
\nStructured Sections:
  Member texts: 2.57 ± 2.09
  Non-member texts: 2.57 ± 2.13
\nMedical Abbreviations (CAPS):
  Member texts: 0.00 ± 0.00
  Non-member texts: 0.00 ± 0.00
\nMedication Patterns:
  Member texts: 0.55 ± 1.47
  Non-member texts: 0.58 ± 1.52


alt.VConcatChart(...)

In [16]:
# N-gram Rarity Analysis (Key for Membership Inference)
from collections import defaultdict, Counter
import math

def analyze_ngram_rarity(df, n_values=[2, 3, 4]):
    """Analyze n-gram frequency patterns that might indicate memorization"""
    
    df = df.copy()
    
    # Get all texts for computing global frequencies
    all_texts = df['text'].str.lower().tolist()
    
    for n in n_values:
        print(f"\\nAnalyzing {n}-grams...")
        
        # Count all n-grams in the dataset
        all_ngrams = []
        for text in all_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            all_ngrams.extend(ngrams)
        
        ngram_counts = Counter(all_ngrams)
        total_ngrams = len(all_ngrams)
        
        # For each text, calculate n-gram rarity scores
        rarity_scores = []
        rare_ngram_counts = []
        
        for text in df['text'].str.lower():
            words = text.split()
            if len(words) < n:
                rarity_scores.append(0)
                rare_ngram_counts.append(0)
                continue
                
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            
            # Calculate average rarity (negative log frequency)
            text_rarity_score = 0
            rare_count = 0
            
            for ngram in ngrams:
                frequency = ngram_counts[ngram] / total_ngrams
                rarity = -math.log(frequency)
                text_rarity_score += rarity
                
                # Count rare n-grams (appear ≤ 2 times in entire dataset)
                if ngram_counts[ngram] <= 2:
                    rare_count += 1
            
            avg_rarity = text_rarity_score / len(ngrams) if ngrams else 0
            rarity_scores.append(avg_rarity)
            rare_ngram_counts.append(rare_count)
        
        df[f'{n}gram_avg_rarity'] = rarity_scores
        df[f'{n}gram_rare_count'] = rare_ngram_counts
        
        # Print top rare n-grams by class
        member_texts = df[df['is_member'] == 1]['text'].str.lower().tolist()
        non_member_texts = df[df['is_member'] == 0]['text'].str.lower().tolist()
        
        # Get n-grams from member vs non-member texts
        member_ngrams = []
        non_member_ngrams = []
        
        for text in member_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            member_ngrams.extend(ngrams)
            
        for text in non_member_texts:
            words = text.split()
            ngrams = [' '.join(words[i:i+n]) for i in range(len(words)-n+1)]
            non_member_ngrams.extend(ngrams)
        
        member_ngram_counts = Counter(member_ngrams)
        non_member_ngram_counts = Counter(non_member_ngrams)
        
        # Find n-grams that appear much more in one class than the other
        member_specific = []
        non_member_specific = []
        
        for ngram, count in member_ngram_counts.items():
            if count >= 3:  # Must appear at least 3 times
                other_count = non_member_ngram_counts.get(ngram, 0)
                if other_count == 0 or count / (other_count + 1) >= 3:  # 3x more frequent
                    member_specific.append((ngram, count, other_count))
        
        for ngram, count in non_member_ngram_counts.items():
            if count >= 3:
                other_count = member_ngram_counts.get(ngram, 0)
                if other_count == 0 or count / (other_count + 1) >= 3:
                    non_member_specific.append((ngram, count, other_count))
        
        print(f"  Class-specific {n}-grams:")
        print(f"    Member-specific: {len(member_specific)} unique {n}-grams")
        if member_specific:
            print(f"      Top examples: {sorted(member_specific, key=lambda x: x[1], reverse=True)[:3]}")
        
        print(f"    Non-member-specific: {len(non_member_specific)} unique {n}-grams")
        if non_member_specific:
            print(f"      Top examples: {sorted(non_member_specific, key=lambda x: x[1], reverse=True)[:3]}")
    
    return df

# Perform n-gram analysis
print("N-gram Rarity Analysis (Memorization Detection):")
print("=" * 60)

train_df_ngrams = analyze_ngram_rarity(train_df)

# Compare n-gram rarity by class
member_ngrams = train_df_ngrams[train_df_ngrams['is_member'] == 1]
non_member_ngrams = train_df_ngrams[train_df_ngrams['is_member'] == 0]

print(f"\\nN-gram Rarity Comparison:")
for n in [2, 3, 4]:
    print(f"\\n{n}-gram Analysis:")
    print(f"  Member avg rarity: {member_ngrams[f'{n}gram_avg_rarity'].mean():.3f} ± {member_ngrams[f'{n}gram_avg_rarity'].std():.3f}")
    print(f"  Non-member avg rarity: {non_member_ngrams[f'{n}gram_avg_rarity'].mean():.3f} ± {non_member_ngrams[f'{n}gram_avg_rarity'].std():.3f}")
    
    print(f"  Member rare {n}-grams per text: {member_ngrams[f'{n}gram_rare_count'].mean():.2f}")
    print(f"  Non-member rare {n}-grams per text: {non_member_ngrams[f'{n}gram_rare_count'].mean():.2f}")

# Visualize n-gram patterns
ngram_comparison_data = pd.concat([
    pd.DataFrame({
        'bigram_rarity': member_ngrams['2gram_avg_rarity'],
        'trigram_rarity': member_ngrams['3gram_avg_rarity'], 
        'fourgram_rarity': member_ngrams['4gram_avg_rarity'],
        'rare_bigrams': member_ngrams['2gram_rare_count'],
        'class': 'Member'
    }),
    pd.DataFrame({
        'bigram_rarity': non_member_ngrams['2gram_avg_rarity'],
        'trigram_rarity': non_member_ngrams['3gram_avg_rarity'],
        'fourgram_rarity': non_member_ngrams['4gram_avg_rarity'],
        'rare_bigrams': non_member_ngrams['2gram_rare_count'],
        'class': 'Non-Member'
    })
]).reset_index(drop=True)

# Create n-gram rarity visualization
bigram_chart = alt.Chart(ngram_comparison_data).mark_boxplot().encode(
    x=alt.X('class:N'),
    y=alt.Y('bigram_rarity:Q', title='Average Bigram Rarity'),
    color=alt.Color('class:N')
).properties(title='Bigram Rarity', width=200, height=180)

trigram_chart = alt.Chart(ngram_comparison_data).mark_boxplot().encode(
    x=alt.X('class:N'),
    y=alt.Y('trigram_rarity:Q', title='Average Trigram Rarity'),
    color=alt.Color('class:N')
).properties(title='Trigram Rarity', width=200, height=180)

(bigram_chart | trigram_chart).show()

N-gram Rarity Analysis (Memorization Detection):
\nAnalyzing 2-grams...
  Class-specific 2-grams:
    Member-specific: 38249 unique 2-grams
      Top examples: [('right fallopian', 37, 11), ('lacrimal sac', 35, 9), ('resected using', 31, 8)]
    Non-member-specific: 39010 unique 2-grams
      Top examples: [('mr images', 51, 13), ('our pain', 38, 9), ('an international', 37, 10)]
\nAnalyzing 3-grams...
  Class-specific 3-grams:
    Member-specific: 59378 unique 3-grams
      Top examples: [('hilar lymph node', 37, 10), ('brain. the patient', 36, 11), ('and renal failure.', 33, 10)]
    Non-member-specific: 61068 unique 3-grams
      Top examples: [('effect. the patient', 45, 14), ('during the 1-year', 34, 7), ('taken at the', 33, 6)]
\nAnalyzing 4-grams...
  Class-specific 4-grams:
    Member-specific: 52466 unique 4-grams
      Top examples: [('the brain. the patient', 31, 6), ('showed no abnormalities, and', 30, 6), ('but the patient remained', 27, 8)]
    Non-member-specific: 54972 

alt.HConcatChart(...)

In [17]:
# 10. Text Similarity and Clustering Analysis

def analyze_text_similarity(df, sample_size=1000):
    """Analyze similarity patterns between member and non-member texts"""
    
    # Sample for computational efficiency
    sampled_df = df.sample(n=min(sample_size, len(df)), random_state=42)
    
    # Vectorize texts
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
    tfidf_matrix = vectorizer.fit_transform(sampled_df['text'])
    
    # Calculate pairwise similarities
    similarity_matrix = cosine_similarity(tfidf_matrix)
    
    # Analyze intra-class vs inter-class similarities
    member_indices = sampled_df[sampled_df['is_member'] == 1].index.tolist()
    non_member_indices = sampled_df[sampled_df['is_member'] == 0].index.tolist()
    
    # Map global indices to matrix indices
    index_mapping = {idx: i for i, idx in enumerate(sampled_df.index)}
    member_matrix_indices = [index_mapping[idx] for idx in member_indices if idx in index_mapping]
    non_member_matrix_indices = [index_mapping[idx] for idx in non_member_indices if idx in index_mapping]
    
    # Calculate average similarities
    member_intra_similarities = []
    non_member_intra_similarities = []
    inter_class_similarities = []
    
    # Intra-class similarities (member-member)
    for i in range(len(member_matrix_indices)):
        for j in range(i+1, len(member_matrix_indices)):
            sim = similarity_matrix[member_matrix_indices[i], member_matrix_indices[j]]
            member_intra_similarities.append(sim)
    
    # Intra-class similarities (non-member to non-member)
    for i in range(len(non_member_matrix_indices)):
        for j in range(i+1, len(non_member_matrix_indices)):
            sim = similarity_matrix[non_member_matrix_indices[i], non_member_matrix_indices[j]]
            non_member_intra_similarities.append(sim)
    
    # Inter-class similarities (member to non-member)
    for i in member_matrix_indices:
        for j in non_member_matrix_indices:
            sim = similarity_matrix[i, j]
            inter_class_similarities.append(sim)
    
    # Calculate statistics
    member_intra_mean = np.mean(member_intra_similarities) if member_intra_similarities else 0
    non_member_intra_mean = np.mean(non_member_intra_similarities) if non_member_intra_similarities else 0
    inter_class_mean = np.mean(inter_class_similarities) if inter_class_similarities else 0
    
    # PCA for 2D visualization
    pca = PCA(n_components=2)
    embeddings_2d = pca.fit_transform(tfidf_matrix.toarray())
    
    # Create visualization data
    viz_df = sampled_df.copy()
    viz_df['pc1'] = embeddings_2d[:, 0]
    viz_df['pc2'] = embeddings_2d[:, 1]
    viz_df['class_label'] = viz_df['is_member'].map({1: 'Member', 0: 'Non-Member'})
    
    return {
        'member_intra_mean': member_intra_mean,
        'non_member_intra_mean': non_member_intra_mean,
        'inter_class_mean': inter_class_mean,
        'member_intra_similarities': member_intra_similarities,
        'non_member_intra_similarities': non_member_intra_similarities,
        'inter_class_similarities': inter_class_similarities,
        'viz_df': viz_df
    }

print("Text Similarity Analysis:")
print("=" * 40)

# Analyze similarity patterns
similarity_results = analyze_text_similarity(train_df, sample_size=1000)

print(f"\\nAverage Cosine Similarities:")
print(f"  Member-to-Member: {similarity_results['member_intra_mean']:.4f}")
print(f"  Non-Member-to-Non-Member: {similarity_results['non_member_intra_mean']:.4f}")
print(f"  Member-to-Non-Member: {similarity_results['inter_class_mean']:.4f}")

print(f"\\nSimilarity Statistics:")
print(f"  Member intra-class std: {np.std(similarity_results['member_intra_similarities']):.4f}")
print(f"  Non-member intra-class std: {np.std(similarity_results['non_member_intra_similarities']):.4f}")
print(f"  Inter-class std: {np.std(similarity_results['inter_class_similarities']):.4f}")

# Visualize text embeddings in 2D space
scatter_chart = alt.Chart(similarity_results['viz_df']).mark_circle(size=30, opacity=0.6).encode(
    x=alt.X('pc1:Q', title='First Principal Component'),
    y=alt.Y('pc2:Q', title='Second Principal Component'),
    color=alt.Color('class_label:N', title='Class'),
    tooltip=['id:N', 'class_label:N']
).properties(
    title='Text Similarity Space (PCA Projection)',
    width=500,
    height=400
)

scatter_chart.show()

# Create similarity distribution comparison
similarity_dist_data = []
for sim in similarity_results['member_intra_similarities'][:1000]:  # Limit for performance
    similarity_dist_data.append({'similarity': sim, 'type': 'Member-Member'})
for sim in similarity_results['non_member_intra_similarities'][:1000]:
    similarity_dist_data.append({'similarity': sim, 'type': 'Non-Member-Non-Member'})
for sim in similarity_results['inter_class_similarities'][:2000]:  # Sample inter-class 
    similarity_dist_data.append({'similarity': sim, 'type': 'Member-Non-Member'})

similarity_df = pd.DataFrame(similarity_dist_data)

similarity_hist = alt.Chart(similarity_df).mark_bar(opacity=0.7).encode(
    x=alt.X('similarity:Q', bin=alt.Bin(maxbins=50), title='Cosine Similarity'),
    y=alt.Y('count():Q', title='Frequency'),
    color=alt.Color('type:N', title='Similarity Type'),
    column=alt.Column('type:N', title='Comparison Type')
).properties(
    title='Distribution of Text Similarities',
    width=200,
    height=150
)

similarity_hist.show()

Text Similarity Analysis:
\nAverage Cosine Similarities:
  Member-to-Member: 0.0577
  Non-Member-to-Non-Member: 0.0577
  Member-to-Non-Member: 0.0577
\nSimilarity Statistics:
  Member intra-class std: 0.0331
  Non-member intra-class std: 0.0327
  Inter-class std: 0.0330


alt.Chart(...)

alt.Chart(...)

# Next Steps: Membership Inference Attack

## Key Analyses Completed:
1. **Text Uniqueness & Rarity** - Look for memorized unique phrases
2. **Medical Domain Patterns** - Discharge note structure differences  
3. **N-gram Rarity** - Find memorized sequences (most important)
4. **Text Similarity** - Clustering and similarity patterns

## Potential Attack Strategies:
1. **Loss-based detection** - Use model perplexity/loss on texts
2. **Confidence-based** - Analyze model confidence patterns
3. **N-gram memorization** - Detect memorized rare sequences
4. **Embedding analysis** - Use text representations for detection

## Papers to Explore:
- "Do Membership Inference Attacks Work on Large Language Models?"
- "What Hard Tokens Reveal: Exploiting Low-confidence Tokens"
- Focus on **perplexity** and **confidence calibration** methods